# 02 — Embeddings as an inspectable data product

Embeddings are versioned feature artifacts, not magic coordinates. This lesson discovers accepted immutable releases from a manifest, loads one bounded shard, keeps vectors aligned with IDs, computes cosine diagnostics, and draws colorblind-safe projections. Fixture vectors are synthetic and illustrative; proximity is not functional equivalence.


In [ ]:
from __future__ import annotations
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from manage_db.public_notebooks import (
    PUBLIC_KG_ROOT,
    build_public_fixture,
    parquet_catalog,
    read_bounded_parquet,
)

MODE = os.environ.get("JOUVENCE_DATA_MODE", "fixture")
BILLING_PROJECT = os.environ.get("JOUVENCE_BILLING_PROJECT")
CACHE = REPO_ROOT / "artifacts" / "cache" / "public-notebooks"
CACHE.mkdir(parents=True, exist_ok=True)
KG_ROOT = build_public_fixture(CACHE / "kg-fixture") if MODE == "fixture" else PUBLIC_KG_ROOT
print({"mode": MODE, "kg_root": str(KG_ROOT), "bounded": True})


## Discover accepted immutable releases instead of guessing paths

A consumable embedding release declares state, immutability, modality, model, source hashes, coverage, license, and shard paths. Fixture mode provides a synthetic accepted manifest. Live mode requires `JOUVENCE_EMBEDDING_MANIFEST_URI` pointing to an explicitly reviewed manifest; there is no mutable latest-path default.


In [ ]:
from manage_db.public_notebooks import discover_embedding_releases
if MODE == "fixture":
    EMBEDDING_MANIFEST = Path(KG_ROOT) / "features" / "embeddings" / "manifest.json"
else:
    EMBEDDING_MANIFEST = os.environ.get("JOUVENCE_EMBEDDING_MANIFEST_URI")
    if not EMBEDDING_MANIFEST:
        raise RuntimeError("Set JOUVENCE_EMBEDDING_MANIFEST_URI to an accepted immutable release manifest")
releases = discover_embedding_releases(EMBEDDING_MANIFEST, billing_project=BILLING_PROJECT)
display(releases)


### Interpretation

Discovery filters out staged, rejected, or mutable artifacts by default. A path being readable does not make its science accepted. In particular, no rejected or staged genomic-gene candidate is presented as canonical here.


In [ ]:
required_release_fields = {"release_id", "state", "immutable", "modality", "license", "coverage", "shard_uri"}
assert required_release_fields.issubset(releases.columns)
assert releases["state"].eq("accepted").all() and releases["immutable"].eq(True).all()
print("accepted immutable shards:", len(releases))


### Checkpoint

Before loading vectors, inspect the release state, immutable identity, encoder revision, preprocessing, license, coverage denominator, and task-specific leakage policy.


## Choose modality for the scientific question

Gene text, genomic-gene sequence, transcript nucleotide, protein sequence, molecule structure, and ontology/text embeddings encode different source surfaces. Text captures documentation semantics; sequence captures molecular patterns; structure captures chemistry. Keep modalities separate and fuse downstream only with an explicit policy.


In [ ]:
modality_guide = pd.DataFrame([
    ("gene text", "gene summaries and ontology-linked prose", "annotation circularity"),
    ("genomic gene", "genomic sequence context", "length/species/build effects"),
    ("transcript nucleotide", "isoform cDNA or UTR", "isoform selection"),
    ("protein sequence", "amino-acid sequence", "paralogy and domain composition"),
    ("molecule structure", "SMILES/graph/fingerprint", "stereochemistry and salts"),
    ("ontology/text", "labels, definitions, hierarchy prose", "label leakage"),
], columns=["modality", "signal", "risk"])
display(modality_guide)


### Interpretation

Text similarity and sequence modality similarity are not interchangeable. Two genes can share textual disease annotations without sequence homology, or share domains without equivalent biological roles. License and source-release compatibility may also differ by modality.


In [ ]:
selected = releases.query("modality == 'text'").copy()
if selected.empty:
    raise RuntimeError("No accepted immutable text release is available in the selected manifest")
release = selected.iloc[0]
print(release[["release_id", "modality", "license", "coverage", "shard_uri"]].to_dict())


### Checkpoint

State the modality before interpreting a neighbor list. Do not call a generic vector simply a gene embedding when the source is specifically text, sequence, structure, or an ontology description.


## Load a bounded shard and preserve row alignment

The loader caps rows and validates required columns. Matrix extraction checks dimensions and finite values, while returning metadata in exactly the same row order. Stable ID lookup then maps biological identifiers to matrix positions without positional guessing.


In [ ]:
from manage_db.public_notebooks import (
    extract_embedding_matrix,
    load_bounded_embedding_sample,
    lookup_embedding_id,
)
sample = load_bounded_embedding_sample(release["shard_uri"], limit=100, billing_project=BILLING_PROJECT)
matrix, embedding_metadata = extract_embedding_matrix(sample)
print({"rows": matrix.shape[0], "dimensions": matrix.shape[1], "metadata_rows": len(embedding_metadata)})
display(embedding_metadata.head())


### Interpretation

Nonfinite vectors fail closed because they poison distances and projections. Zero vectors remain detectable through their norm and should be excluded or handled explicitly; they must not silently masquerade as missing source-backed signal.


In [ ]:
query_id = "ENSG00000012048"
query_position = lookup_embedding_id(embedding_metadata, query_id)
assert embedding_metadata.iloc[query_position]["node_id"] == query_id
print({"query_id": query_id, "row_position": query_position, "vector": matrix[query_position].round(3).tolist()})


### Checkpoint

Always carry aligned metadata—ID, node type, model, and release—with the matrix. A bare NumPy array cannot support auditable biological interpretation.


## Join labels and diagnose coverage, missingness, and norms

Coverage is defined against a named population, not against rows that happened to load. We join stable IDs to fixture labels and compare represented IDs with the selected node tables. Vector norms reveal zero rows and scale outliers before cosine or projection analysis.


In [ ]:
node_frames = []
for node_type in ("gene", "disease", "molecule"):
    frame = read_bounded_parquet(f"{str(KG_ROOT).rstrip('/')}/nodes/{node_type}.parquet", limit=100, billing_project=BILLING_PROJECT)
    frame = frame[["id", "name"]].assign(node_type=node_type)
    node_frames.append(frame)
labels = pd.concat(node_frames, ignore_index=True).rename(columns={"id": "node_id", "name": "label"})
embedding_metadata = embedding_metadata.merge(labels, on=["node_id", "node_type"], how="left", validate="one_to_one")
embedding_metadata["vector_norm"] = __import__("numpy").linalg.norm(matrix, axis=1)
display(embedding_metadata[["node_id", "node_type", "label", "vector_norm"]])


### Interpretation

Missing labels indicate a failed or incomplete identity join, while missing vectors indicate coverage gaps. Neither should be repaired with fabricated biological vectors. A model-side fallback may support execution only when it is declared as fallback rather than source-backed coverage.


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(embedding_metadata["node_id"], embedding_metadata["vector_norm"], color="#0072B2")
ax.set(title="Fixture vector-norm diagnostic", ylabel="L2 norm", xlabel="stable entity ID")
ax.tick_params(axis="x", rotation=75)
fig.tight_layout()
assert len(ax.patches) == len(embedding_metadata)
plt.show()


### Checkpoint

The plot must have one bar per aligned row. Fixture norms are synthetic diagnostics, not evidence that one entity is biologically stronger or better represented.


## Compute pair similarity and nearest neighbors

Cosine compares vector direction after normalization. It is useful for retrieval within one compatible release, but it ignores magnitude and inherits encoder, corpus, preprocessing, and coverage biases. We inspect both one pair and the full bounded similarity distribution.


In [ ]:
from manage_db.public_notebooks import cosine_neighbors, pairwise_cosine
similarity = pairwise_cosine(matrix)
brca1 = lookup_embedding_id(embedding_metadata, "ENSG00000012048")
brca2 = lookup_embedding_id(embedding_metadata, "ENSG00000139618")
print("BRCA1–BRCA2 cosine:", round(float(similarity[brca1, brca2]), 4))
neighbors = cosine_neighbors(matrix, embedding_metadata, "ENSG00000012048", limit=5)
display(neighbors[["node_id", "node_type", "label", "cosine_similarity"]])


### Interpretation

High proximity is not functional equivalence, causal interaction, interchangeability, or treatment evidence. It means only that this encoder placed the selected source payloads near one another under cosine distance.


In [ ]:
import numpy as np
upper = similarity[np.triu_indices_from(similarity, k=1)]
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(upper, bins=min(10, max(3, len(upper) // 3)), color="#009E73", edgecolor="white")
ax.set(title="Bounded pairwise cosine distribution", xlabel="cosine similarity", ylabel="pair count")
assert len(ax.patches) > 0 and np.isfinite(upper).all()
fig.tight_layout()
plt.show()


### Checkpoint

Inspect the distribution before choosing a neighbor cutoff. Small fixture distributions are illustrative and should never be used as production thresholds.


## Project vectors with UMAP or a deterministic PCA fallback

Two-dimensional projections are lossy visual summaries. The helper uses UMAP when installed and otherwise a deterministic documented PCA fallback. We use a colorblind-safe palette, label axes and method, and annotate a few stable IDs rather than implying exact cluster boundaries.


In [ ]:
from manage_db.public_notebooks import project_embedding_matrix
coordinates, projection_method = project_embedding_matrix(matrix, method="auto", random_state=17)
projection = embedding_metadata.copy()
projection[["x", "y"]] = coordinates
print({"projection": projection_method, "shape": coordinates.shape, "fallback": "pca" if projection_method == "pca" else None})


### Interpretation

UMAP preserves selected local neighborhoods stochastically; PCA preserves directions of greatest linear variance. Their axes are not biological dimensions. The PCA fallback makes clean environments executable and reproducible without pretending the methods are equivalent.


In [ ]:
palette = {"gene": "#0072B2", "disease": "#D55E00", "molecule": "#009E73"}
fig, ax = plt.subplots(figsize=(7, 5))
for node_type, group in projection.groupby("node_type", sort=True):
    ax.scatter(group["x"], group["y"], label=node_type, color=palette[node_type], s=60, alpha=0.85)
annotation_offsets = {
    "ENSG00000139618": (-38, 8),
    "ENSG00000012048": (5, -13),
    "ENSG00000141510": (5, 5),
    "ENSG00000146648": (5, -11),
}
for row in projection.head(4).itertuples():
    ax.annotate(
        row.label or row.node_id,
        (row.x, row.y),
        xytext=annotation_offsets.get(row.node_id, (5, 5)),
        textcoords="offset points",
        fontsize=8,
    )
ax.set(title=f"Synthetic fixture embedding — {projection_method.upper()}", xlabel="component 1", ylabel="component 2")
ax.legend(title="entity type")
ax.margins(x=0.15, y=0.15)
assert len(ax.collections) == projection["node_type"].nunique()
fig.tight_layout()
plt.show()


### Checkpoint

For larger interactive work, replace annotations with hover labels while retaining stable IDs and release metadata. Continue to Notebook 03 for source evidence; never treat visual clusters as validated function classes.
